In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when, isnan

catalog_name = 'automobilerepair'

In [0]:
df = spark.read.table("automobilerepair.bronze.stg_ns_budget")

In [0]:
row_count = df.count()
row_count

In [0]:
#Finding duplicates based on the aggregation of store_id and month since store has budget for each month over year
duplicates_df = df.groupBy("ns_store_id", "month") \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1)

if duplicates_df.count() > 0:
    df = df.dropDuplicates(["ns_store_id", "month"])

In [0]:
print("\nNULL VALUE ANALYSIS")

null_counts = df.select([ count(when(col(c).isNull(), c)).alias(c) for c in df.columns ])
print("Null counts by column:")
display(null_counts)

In [0]:
df = df.withColumn('currency', f.lit('INR'))

display(df.select('currency', 'ns_store_id').limit(5))

In [0]:
#Snake case handling
df = df.withColumnRenamed("_modified", "modified")

#Date conversion
df = df.withColumn("modified", f.to_date(col("modified")))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_budget")